浮士德 LoRA 基线训练

先在“运行时 → 更改运行时类型”选择 NVIDIA GPU。建议24GB或以上显存；16GB仅尝试，试跑失败不会删减原文。随后“运行全部”。这是准备好的训练入口，不是已完成GPU验证的模型。公开仓库与公开底模不要求你粘贴API密钥。

默认将检查点保存至你授权的Google Drive。实际算力是否收费取决于你选择的运行时；本笔记本不会购买资源。

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPOSITORY = "https://github.com/Loong-C/Faust.git"
REF = "training/lora-v0.1.0"
REPO_DIR = Path("/content/FaustTraining")
SAVE_TO_DRIVE = True
RUN_NAME = "faust-lora-v0.1.0"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_ROOT = Path("/content/drive/MyDrive/FaustTraining")
else:
    RUN_ROOT = Path("/content/faust_runs")
    print("警告：本地运行时被回收后会丢失检查点；请及时下载结果。")
RUN_DIR = RUN_ROOT / RUN_NAME
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF, REPOSITORY, str(REPO_DIR)], check=True)
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError("目标目录不是Git仓库；请换一个目录，不覆盖现有文件。")
    dirty = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True)
    if dirty.strip():
        raise RuntimeError("工作目录有本地修改，请先保存；不会自动重置。")
    current = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
    if current != REF:
        raise RuntimeError("已有目录不在指定训练分支；请换新的REPO_DIR。")
print("代码版本：", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())
print("结果目录：", RUN_DIR)


安装、试跑与正式训练

本单元会在独立虚拟环境安装固定依赖，测试最长样本的两步训练，再从原始底模开始正式训练。正常中断后重新运行会使用完整检查点；不要更改RUN_NAME。要做新的超参数实验时才更换RUN_NAME。

In [ ]:
env = dict(os.environ)
env["FAUST_OUTPUT"] = str(RUN_DIR)
subprocess.run(["bash", "training/run.sh"], cwd=REPO_DIR, env=env, check=True)


训练后比较与打包

默认生成3个原创情景的底模/LoRA对照，训练集与参考材料不变。德文比较结果附中文情景说明，可以将文件交回对话分析。备份包不含底模，也不含用于恢复的优化器检查点；完整续训记录仍在Drive目录中。

In [ ]:
PY = REPO_DIR / ".venv-faust/bin/python"
subprocess.run([str(PY), "training/compare.py", "--run", str(RUN_DIR), "--limit", "3"], cwd=REPO_DIR, check=True)
import zipfile
ARCHIVE = RUN_ROOT / (RUN_NAME + "-adapter-and-reports.zip")
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (RUN_DIR / "adapter", RUN_DIR / "comparisons"):
        for p in sorted(folder.rglob("*")):
            if p.is_file():
                z.write(p, p.relative_to(RUN_DIR))
    for name in ("run_manifest.json", "preflight.json", "metrics.json", "base_validation.json", "trainer_state.json", "pip_freeze.txt"):
        p = RUN_DIR / name
        if p.is_file():
            z.write(p, p.name)
print("已保存：", ARCHIVE)
from google.colab import files
files.download(str(ARCHIVE))
